In [1]:
#Referenz ⇒ Canonical Form 
#pdb ID in der pdb suchen ⇒ FASTA sequence file downloaden  
#bei SAbPred: SCALOP in Submission form die Datei hochladen 
#results: für jedes CDR (H1, H2, L1, L2, L3) erkennt er CDR Sequenz aus der gesamten Sequenz (mit Antibody und antigen) und gibt einem canonical form und median structure 
#(L1-11-A → Canonical form A für eine L1-Schleife mit 11 Aminosäuren)
#muss man für alle unsere pdb Einträge machen und dann canonical cluster nochmal im code definieren (also die pdb Einträge zuordnen)
#dann der vergleich mit V-measure

In [ ]:
#fasta dateien downloaden

import requests      #Modul zum Herunterladen von Daten aus dem Internet
import os            #Modul für Dateipfade und Ordnerverwaltung

def download_fasta(pdb_id, outdir="fasta_files"):
    """
    Lädt die FASTA-Sequenzdatei für einen gegebenen PDB-Eintrag
    von der RCSB PDB-Website herunter und speichert sie lokal.

    Parameter:
    - pdb_id: z.B. "1abc" (Groß-/Kleinschreibung egal)
    - outdir: Zielordner, in dem die FASTA-Dateien gespeichert werden

    Rückgabe:
    - Pfad zur gespeicherten FASTA-Datei (oder None bei Fehler)
    """

    #URL zur FASTA-Datei auf der rcsb.org-Website (liefert alle Chains)
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}/display"

    #HTTP-GET-Request an die URL schicken
    response = requests.get(url)

    #Prüfen ob der Download erfolgreich war (Statuscode 200 = OK)
    if response.status_code == 200:

        #Zielordner anlegen, falls er noch nicht existiert
        os.makedirs(outdir, exist_ok=True)

        #Speicherpfad für die Datei zusammensetzen
        fasta_path = os.path.join(outdir, f"{pdb_id}.fasta")

        #Inhalt in Datei schreiben
        with open(fasta_path, "w") as f:
            f.write(response.text)

        #Pfad zur fertigen Datei zurückgeben
        return fasta_path
        

    else:
        #Fehlerausgabe, falls Download fehlgeschlagen
        print(f"Fehler beim Herunterladen von {pdb_id} (Status: {response.status_code})")
        return None
    
    
    

In [3]:
#hochladen auf sabpred und extraktion der canonical forms

In [4]:
import time
import pandas as pd
%pip install selenium webdriver-manager

# Automatisierung mit Selenium. Anforderung: selenium und webdriver-manager installiert
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Hilft, automatisch passenden ChromeDriver zu finden
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from selenium.webdriver.common.action_chains import ActionChains
#pdb ids aus unserer geflterten datei laden
df = pd.read_csv("data/ab_ag_annotated.tsv", sep="\t").drop_duplicates(subset = ["pdb", "CDR_H1", "CDR_H2", "CDR_L1", "CDR_L2", "CDR_L3"], ignore_index = True)
pdb_ids = df["pdb"].unique() #nur eindeutige PDB-IDs extrahieren

#automatisierten brwoser starten
# Chrome-Optionen: "headless" = läuft ohne sichtbares Fenster
options = webdriver.ChromeOptions()
options.add_argument('--headless')  # unsichtbar im Hintergrund => hab es sichtbar aus probiert => der link zur startseite öffnet sich auf jeden fall

# Browser starten mit automatisch installiertem Treiber (=Schnittstelle zwischen python skript und browser)
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

results = []  # Hier wird jedes Canonical-Form-Ergebnis als dict gespeichert

#Iteration über PDB-IDs
for pdb_id in pdb_ids:
    download_fasta(pdb_id)

    # Gehe zur sappred-Webseite
    driver.get("https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabpred")
    

   #warten damit auf seite auch alles geladen ist
    time.sleep(5)

    # Suche Link zu SCALOP über XPATH und überprüfe, ob Element vorhanden ist
    try:
        scalop_link = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.LINK_TEXT, "SCALOP"))
        )
        driver.execute_script("arguments[0].scrollIntoView();", scalop_link)
        scalop_link.click()
        print("SCALOP Link erfolgreich geklickt!")
    except Exception as e:
        print("Konnte SCALOP Link nicht klicken:", e)
    
    #navigation auf der scalop seite
    # Warte bis Formular geladen ist
    WebDriverWait(driver, 20).until(
    EC.presence_of_element_located((By.XPATH, '//input[@type="file"]'))
    )

    # Scrolle nach unten
    actions = ActionChains(driver)
    actions.move_to_element(driver.find_element(By.XPATH, '//input[@type="file"]')).perform()

    # Lade FASTA-Datei hoch aber mit wartezeit damit die seite auch wirklich geladen ist
    file_input = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.XPATH, '//input[@type="file"]'))
    )
    
    file_input.send_keys('fasta_files\\9ds2.fasta') #schickt den lokalen Pfad der Datei an dieses Feld

    # Wähle Chothia bei Numbering Scheme
    chothia_radio = driver.find_element(By.XPATH, '//input[@type="radio" and @value="chothia"]')
    chothia_radio.click()

    # Wähle Chothia bei CDR Definition
    cdr_chothia_radio = driver.find_element(By.XPATH, '//input[@type="radio" and @value="Chothia"]')
    cdr_chothia_radio.click()

    # Klicke Assign-Button
    assign_button = driver.find_element(By.XPATH, '//button[contains(text(), "Assign")]')
    assign_button.click()

    

    #auf ergebnisse warten
    try:
    # Warten bis Ergebnisse (Tabelle) erscheinen (max. 20 Sekunden)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, '//table')))

    # Alle Zeilen (außer Header) in der Ergebnistabelle erfassen
        rows = driver.find_elements(By.XPATH, '//table//tr[position()>1]')      

        #für jede zeile relevante infos extrahieren
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, 'td')
            if len(cols) >= 5:
                chain = cols[0].text.strip()
                cdr = cols[1].text.strip()
                length = cols[2].text.strip()
                canonical_form = cols[3].text.strip()

                # Ergebnis in Dictionary speichern
                results.append({
                "PDB_ID": pdb_id,
                "Chain": chain,
                "CDR": cdr,
                "Length": length,
                "Canonical_Form": canonical_form })

    except Exception as e:
            print(f"Fehler bei {pdb_id}: {e}")
            continue  # Wenn etwas schiefläuft, einfach zur nächsten PDB-ID springen

#browser schließen und ergbnisse speichern
# Browser schließen
driver.quit()

# Ergebnisse in DataFrame umwandeln und abspeichern
df_results = pd.DataFrame(results)
df_results.to_csv("scalop_canonical_forms.csv", index=False)

SCALOP Link erfolgreich geklickt!


WebDriverException: Message: unknown error: path is not absolute: fasta_files\9ds2.fasta
  (Session info: chrome=138.0.7204.50)
Stacktrace:
	GetHandleVerifier [0x0x674493+62419]
	GetHandleVerifier [0x0x6744d4+62484]
	(No symbol) [0x0x4b2133]
	(No symbol) [0x0x4c100d]
	(No symbol) [0x0x4f3ae3]
	(No symbol) [0x0x51f46c]
	(No symbol) [0x0x4f0054]
	(No symbol) [0x0x51f6e4]
	(No symbol) [0x0x54087a]
	(No symbol) [0x0x51f266]
	(No symbol) [0x0x4ee852]
	(No symbol) [0x0x4ef6f4]
	GetHandleVerifier [0x0x8e4773+2619059]
	GetHandleVerifier [0x0x8dfb8a+2599626]
	GetHandleVerifier [0x0x69b03a+221050]
	GetHandleVerifier [0x0x68b2b8+156152]
	GetHandleVerifier [0x0x691c6d+183213]
	GetHandleVerifier [0x0x67c378+94904]
	GetHandleVerifier [0x0x67c502+95298]
	GetHandleVerifier [0x0x66765a+9626]
	BaseThreadInitThunk [0x0x76ac5d49+25]
	RtlInitializeExceptionChain [0x0x774dd09b+107]
	RtlGetAppContainerNamedObjectPath [0x0x774dd021+561]


In [ ]:
#überprüfen ob scalop zu finden ist in diesem link 
#wartezeit damit dynamische inhalte zeit haben geladen zu werden
import time

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
# Gehe zur Hauptseite
driver.get("https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabpred")

# Warte 5 Sekunden für vollständiges Laden der Seite
time.sleep(5)

# HTML-Code ausgeben, um zu prüfen, ob der Link im DOM ist
#html = driver.page_source
#print(html)

# Suche Link zu SCALOP über XPATH und überprüfe, ob Element vorhanden ist
try:
    scalop_link = WebDriverWait(driver, 20).until(
        EC.element_to_be_clickable((By.LINK_TEXT, "SCALOP"))
    )
    driver.execute_script("arguments[0].scrollIntoView();", scalop_link)
    scalop_link.click()
    print("SCALOP Link erfolgreich geklickt!")
except Exception as e:
    print("Konnte SCALOP Link nicht klicken:", e)

elements = driver.find_elements(By.XPATH, '//input[@type="file"]')
print(f"Gefundene Elemente: {len(elements)}")
for e in elements:
    print(e.get_attribute('outerHTML'))

    #es gibt also 2 elemnte wir müssen das roichtige ansteuern 


SCALOP Link erfolgreich geklickt!
Gefundene Elemente: 2
<input type="file" name="fastafile" id="fastafile">
<input type="file" name="framework" id="framework">


In [29]:
#fasta downladen
download_fasta("9ds2")

'fasta_files\\9ds2.fasta'

In [31]:
os.path.exists('fasta_files\\9ds2.fasta')

True